# Groundtruth
Please read the `README.md` file before running this tutorial.

Each sensor frame has a timestamp, groundtruth pose, (4x4 homogeneous transform) wrt a global coordinate frame (ENU), and groundtruth velocity information.
This is true for all sequences except for those that are part of the Boreas test set, where the groundtruth is withheld for a fair leaderboard comparison.
The Boreas-RT leaderboard is based on sequences with provided groundtruth on account of the fewer repetitions of each route.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from pyboreas import BoreasDataset
from pyboreas.vis.vis_utils import vis_sensor_poses
import os.path as osp

root = '/path/to/data/boreas/'
split = None
# AWS: Note: Free Tier SageMaker instances don't have enough storage (25 GB) for 1 sequence (100 GB)
# root = '/home/ec2-user/SageMaker/boreas/'
# split = [['boreas-2021-09-02-11-42', 163059759e6, 163059760e6-1]]

# With verbose=True, the following will print information about each sequence
bd = BoreasDataset(root, split=split, verbose=True)
# Grab the first sequence
seq = bd.sequences[0]

In [ ]:
# Each sensor frame has a timestamp, groundtruth pose
# (4x4 homogeneous transform) wrt a global coordinate frame (ENU),
# and groundtruth velocity information. Unless it's a sequence from the Boreas test set,
# in that case, ground truth poses will be missing.

radar_frame = bd.get_radar(0)
t = radar_frame.timestamp  # timestamp in seconds
T_enu_lidar = radar_frame.pose  # 4x4 homogenous transform [R t; 0 0 0 1]
vbar = radar_frame.velocity  # 6x1 vel in ENU frame [v_se_in_e; w_se_in_e]
varpi = radar_frame.body_rate  # 6x1 vel in sensor frame [v_se_in_s; w_se_in_s]

lidar_frame = bd.get_lidar(0)
t = lidar_frame.timestamp  # timestamp in seconds
T_enu_lidar = lidar_frame.pose  # 4x4 homogenous transform [R t; 0 0 0 1]
vbar = lidar_frame.velocity  # 6x1 vel in ENU frame [v_se_in_e; w_se_in_e]
varpi = lidar_frame.body_rate  # 6x1 vel in sensor frame [v_se_in_s; w_se_in_s]

In [ ]:
# We provide independent groundtruth loading functions if you wish to load the groundtruth yourself
# Note that these are used internally in data loading/evaluation scripts, so don't really need to be used by the end user,
# but are available if you want to load the groundtruth yourself for some reason

# The functions all take in a file path to the ground truth csv and output groundtruth data and corresponding timestamps
from pyboreas.utils.odometry import read_traj_file_gt, read_traj_file_gt2, read_vel_file_gt

# read_traj_file_gt
# This function will output T_v_i, where v is the body frame and i is the inertial frame
# The function takes in an optional T_ab parameter, which applies a T_v_s transform between the sensor frame s
# in which the groundtruth poses are provided and the body frame v in which the poses are output.
# By default, this is set to identity, meaning that the function will output T_s_i
# If desired, the function can be run with dim=2 to output poses projected into pure 2D
lid_poses, times = read_traj_file_gt(osp.join(seq.applanix_root, 'lidar_poses.csv'),  T_ab=np.identity(4), dim=3)
print("Full 3D T_lidar_i:\n", lid_poses[0])
lid_poses_2d, times = read_traj_file_gt(osp.join(seq.applanix_root, 'lidar_poses.csv'),  T_ab=np.identity(4), dim=2)
print("2D T_lidar_i:\n", lid_poses_2d[0])

# read_traj_file_gt2
# This function reads the unmodified ground truth poses from the csv in T_i_s format
lid_poses2, times = read_traj_file_gt2(osp.join(seq.applanix_root, 'lidar_poses.csv'))
# With T_ab set to identity, the output of read_traj_file_gt should be the same as the inverse of the output of read_traj_file_gt2
print("Similarity check: ", np.all(abs(lid_poses2[0] - np.linalg.inv(lid_poses[0])) < 1e-6))

# read_vel_file_gt
# This function reads the ground truth body rate velocity from the csv as a 6x1 vel in sensor frame [v_se_in_s; w_se_in_s]
# You can optionally provide a T_ab transform to transform the velocity into a different sensor frame if desired,
# but by default, this is set to identity and outputs the velocity in the same sensor frame as the csv
# You can also set dim=2 to get the velocity projected into pure 2D
lid_vels, times = read_vel_file_gt(osp.join(seq.applanix_root, 'lidar_poses.csv'), T_ab=np.identity(4), dim=3)
print("Full 3D lidar velocity in sensor frame [v_se_in_s; w_se_in_s]:\n", lid_vels[0])
lid_vels_2d, times = read_vel_file_gt(osp.join(seq.applanix_root, 'lidar_poses.csv'), T_ab=np.identity(4), dim=2)
print("2D lidar velocity in sensor frame [v_se_in_s; w_se_in_s]:\n", lid_vels_2d[0])
# This is the same velocity that you'd get from a given lid_frame.body_rate
print("Vel similarity check: ", np.all(abs(lid_vels[0] - bd.get_lidar(0).body_rate) < 1e-6))

In [ ]:
# We can visualize the groundtruth trajectory of lidar, radar, camera, or aeva (if available) sensors
# Note that the groundtruth is plotted in the inertial frame, so all sensors will generate the same trajectory
fig_lid = vis_sensor_poses(seq, sensor='lidar', figsize=(3, 3))
fig_rad = vis_sensor_poses(seq, sensor='radar', figsize=(3, 3))